Il secondo soft fork molto importante in Bitcoin, dopo SegWit, è stato **Taproot** (2021). 

Taproot ha introdotto un insieme di miglioramenti alla struttura degli script e delle firme, e il ruolo centrale in ciò è stato svolto dall'implementazione delle **Firme di Schnorr**. Schnorr è uno schema di firma digitale considerato molto elegante ed affidabile, con proprietà matematiche particolarmente utili per Bitcoin soprattutto in quanto consente di **aggregare efficientemente più firme** in una singola firma, migliorando la privacy.

Quando Bitcoin fu creato, Satoshi Nakamoto non adottò Schnorr ma come sappiamo implementò ECDSA, lo schema basato sulle curve ellittiche già ampiamente utilizzato in crittografia. La motivazione principale era sicuramente che Schnorr all'epoca era ancora sotto brevetto, e Satoshi voleva evitare qualsiasi complicazione legale.

L'introduzione di Taproot ha beneficiato molto dell'update precedente SegWit, dal momento che quest'ultimo aveva già separato i dati di firma dal corpo principale della transazione spostandoli nella parte **witness**. Questo ha reso più semplice l'introduzione di nuove versioni di script e modi di verificare le condizioni di spesa senza dover modificare in modo invasivo la struttura base delle transazioni.

In particolare Taproot è implementato come una nuova versione di output SegWit, non si entra nei dettagli al riguardo. In questa lezione si introduce in particolare il funzionamento delle firme di Schnorr, e si spiega la questione dell'aggregazione delle firme.

### Firme di Schnorr
Le firme di Schnorr sono uno schema di firma digitale che può essere istanziato su un qualsiasi gruppo (ciclico) in cui **il problema del logaritmo discreto sia computazionalmente difficile**. 

In Bitcoin, Schnorr è applicato al gruppo dei punti della curva ellittica secp256k1. In questo gruppo:
- gli elementi del gruppo sono punti della curva ellittica
- l'operazione del gruppo è la somma di punti
- esiste un generatore pubblico $G$
- l'ordine del generatore è un numero primo $n$, poco inferiore a $2^{256}$. Con ordine del generatore si intende che il sottogruppo generato da $G$ ($\langle G \rangle$) ha esattamente $n$ elementi. Inoltre $nG = O$, dove $O$ è il punto all'infinito, che funge da elemento neutro per la somma di punti, per questo motivo quando prendiamo scalari random non includiamo $0$ e $n$ (O non è una chiave pubblica valida)

In quanto schema di firme, vediamo come funzionano KeyGen, Sign e Verify per Schnorr.

```text
KeyGen():

Public parameters:
    G = generator of the group
    n = order of G

1. choose x uniformly at random from {1, ..., n-1} #(secret key)
2. compute P = xG  #(public key)
3. return (x, P) 
```
Quindi la chiave privata è uno scalare $x$ scelto casualmente, e la chiave pubblica è il punto della curva $P$ ottenuto moltiplicando il generatore $G$ per $x$ secondo l'operazione somma definita per il gruppo.
```text
Sign(m, sk):

1. choose k uniformly at random from {1, ..., n-1}
2. compute R = kG
3. compute P = xG
4. compute h = H(R || P || m)
5. compute s = k + h x mod n
6. return sigma = (R, s)
```
Per firmare un messaggio $m$ con chiave privata $x$, Schnorr prevede di scegliere un nonce casuale $k \in [1, n-1]$, dopodiché si calcola $R = kG$ e poi una challenge tramite hash $h = H(R || P || m)$ (hash della concatenazione di $R$, $P$ e $m$). Infine si calcola la firma $s = k + h x \mod n$. La firma è quindi composta da $R$ e $s$.

La parte più interessante di Schnorr è che la firma $s$ ha una struttura **lineare** $s = k + h x$, e ciò che le consente di nascondere la chiave privata $x$ è l'aggiunta del nonce random $k$ (a chi osserva dall'esterno, $s$ sembra un numero casuale proprio perché $k$ è stato scelto random). 

Quindi qui la sicurezza dipende in modo cruciale (similmente ai ragionamenti fatti per ECDSA) dal fatto che $k$ sia segreto e imprevedibile. Se infatti si firmassero due messaggi diversi con lo stesso $k$, si potrebbe risalire facilmente ad $x$:
$$s_1 = k + h_1 x \mod n \qquad s_2 = k + h_2 x \mod n$$
Sottraendo membro a membro:
$$s_1 - s_2 = (h_1 - h_2) x \mod n \implies x = (s_1 - s_2)(h_1 - h_2)^{-1} \mod n$$
```text
Verify(m, sigma, pk):

1. compute h = H(R || P || m)
2. return sG == R + hP
```

> **Correttezza**:
>
> Vogliamo dimostrare che per ogni firma generata correttamente da Sign, questa sarà accettata da Verify.
>
> **Dimostrazione**:
>
> Da Sign abbiamo $s = k + h x \mod n$. Allora:
> $$sG = (k + h x)G = kG + hxG = R + hP$$
> che è esattamente la condizione che Verify controlla, quindi la firma sarà accettata.
>
> $\blacksquare$

La vera differenza tra Schnorr ed ECDSA non risiede solo nella sua semplicità, ma soprattutto **nel fatto che la sua struttura matematica è lineare**. 

Questo permette in particolare di **dimostrare in modo formale la sicurezza di Schnorr**, mostrando che se un attaccante riesce a produrre firme valide senza conoscere la chiave privata, ciò equivarrebbe a dire che l'attaccante è in grado di risolvere il problema del logaritmo discreto. Sarebbe a dire che **falsificare una firma Schnorr è difficile almeno quanto ricavare la chiave privata $x$ dalla chiave pubblica $P = xG$**.

Al contrario, in ECDSA (dal momento che la struttura della firma non è lineare) non esiste una dimostrazione formale di sicurezza, per questo Schnorr è considerato più elegante e affidabile.

Un altro punto fondamentale legato alla linearità della firma di Schnorr è la possibilità di **aggregare più firme** in una singola firma, e questo è un aspetto cruciale per Bitcoin.

### Aggregazione di firme